# Bike Sharing Project – Step 7 onward with Optuna

> [!NOTE]
> **Experiment, written with Claude.** This notebook repeats step 7 (ensembles) and step 8 (test set) of [02_bike_sharing_project_ai_solution.ipynb](02_bike_sharing_project_ai_solution.ipynb), but tunes the models with **Optuna** instead of `RandomizedSearchCV`. Everything before step 7 is the same as in notebook 02 and is only repeated here in short, so the notebook runs on its own.

**The question:** does a smarter hyperparameter search change the results of step 7 and 8?

## What Optuna does differently

`RandomizedSearchCV` draws every try **at random**, independently of how the earlier tries went. Optuna's default sampler (**TPE**, Tree-structured Parzen Estimator) **learns from the earlier tries**: after a few random start-up tries, it splits them into "good" and "bad" ones and draws the next try from the region where the good ones are more likely than the bad ones. So it spends more of its budget where the scores are low.

**A fair comparison** must change only the search method:
- the **same search ranges** as in notebook 02 for the random forest and XGBoost,
- the **same five folds** (`kfold`) and the same metric (CV RMSE),
- the **same number of tries** for the comparison: notebook 02 used 10 tries for the forest and 15 for XGBoost. Optuna gets more here (30 and 50), but its best score is also recorded after every try, so it can be compared at 10 and 15 tries too. Otherwise the extra tries, not the smarter search, would get the credit.

CatBoost was not tuned at all in notebook 02 (default settings). Here it is tuned as well; that is a second change, and it is reported separately.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, StackingRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# only warnings, not one log line per try
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Steps 2–5 in short (the same as notebook 02)

Load the data, the same train-test split (`random_state=42`), the same `prepare()` function and the same five folds. The explanations are in notebook 02.

In [ ]:
df = pd.read_csv("data/bike_sharing.csv")
df["datetime"] = pd.to_datetime(df["datetime"])

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)


def prepare(data):
    """All manual changes to the data, collected in one place (as in notebook 02)."""
    data = data.copy()
    dt = data["datetime"]

    # 1. the hour 02:00 on the second Sunday of March does not exist (clocks go forward)
    nonexistent = (dt.dt.month == 3) & (dt.dt.dayofweek == 6) & dt.dt.day.between(8, 14) & (dt.dt.hour == 2)
    data = data[~nonexistent]
    dt = data["datetime"]

    # 2. calendar features
    data["hour"] = dt.dt.hour
    data["weekday"] = dt.dt.dayofweek
    data["month"] = dt.dt.month
    data["year"] = dt.dt.year

    # 3. + 4. flags for holidays and working days
    data["is_holiday"] = data["holiday"].notna().astype(int)
    data["is_working_day"] = ((data["weekday"] < 5) & (data["is_holiday"] == 0)).astype(int)

    # 5. drop replaced, redundant and leaking columns
    return data.drop(columns=["holiday", "feels_like", "weather_code", "casual", "member"])


df_train_prep = prepare(df_train)
X_train = df_train_prep.drop(columns=["rentals", "datetime"])
y_train = df_train_prep["rentals"]

# the same five folds as in notebook 02
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

print("X_train:", X_train.shape)

The results of notebook 02 (step 7 and 8), to compare with:

In [ ]:
results_02 = pd.DataFrame({
    "CV RMSE":    [159.5, 140.9, 142.5, 140.1, 137.7],
    "Test RMSE":  [148.3, 132.9, 136.3, 132.2, 130.7],
    "Test MAE":   [92.1, 82.9, 84.9, 80.0, 79.8],
}, index=["Random forest", "XGBoost", "CatBoost", "Voting", "Stacking"])

# best CV RMSE of the RandomizedSearchCV in notebook 02 and its number of tries
random_search_02 = {"Random forest": (159.5, 10), "XGBoost": (140.9, 15)}
results_02

## Step 7 – Ensembles, tuned with Optuna

Two small helpers:
- `cv_rmse()` scores a model with the five folds, exactly like `cross_val_score` in notebook 02.
- `run_study()` runs an Optuna **study**: Optuna calls the `objective` function `n_trials` times, each time with new hyperparameters, and keeps the lowest CV RMSE. `TPESampler(seed=42)` makes the tries the same on every run. The best score after every try is kept for the comparison with the random search.

Inside an `objective`, `trial.suggest_int`, `suggest_float` and `suggest_categorical` play the role of `randint`, `uniform` / `loguniform` and a list in `param_distributions`. `log=True` is the same as `loguniform`.

In [ ]:
studies = {}


def cv_rmse(model):
    """Mean CV RMSE with the same five folds as in notebook 02."""
    return -cross_val_score(model, X_train, y_train, cv=kfold,
                            scoring="neg_root_mean_squared_error").mean()


def run_study(name, objective, n_trials):
    start = time.time()
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials)
    studies[name] = study
    print(f"{name}: best CV RMSE {study.best_value:.1f} after {n_trials} tries "
          f"({(time.time() - start) / 60:.1f} min)")
    print("best parameters:", {k: round(v, 3) if isinstance(v, float) else v
                               for k, v in study.best_params.items()})
    return study

### Random forest

The same three hyperparameters and ranges as the `RandomizedSearchCV` in notebook 02: `max_features` between 0.3 and 1.0, `min_samples_leaf` from 1 to 9, `max_depth` one of `None`, 15, 25.

In [ ]:
def rf_objective(trial):
    model = RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1,
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 9),
        max_depth=trial.suggest_categorical("max_depth", [None, 15, 25]),
    )
    return cv_rmse(model)


rf_study = run_study("Random forest", rf_objective, n_trials=30)

### XGBoost

The same six hyperparameters and ranges as in notebook 02. This is where a smarter search has the most room: six dimensions, and notebook 02 tried only 15 random points in them.

In [ ]:
def xgb_objective(trial):
    model = XGBRegressor(
        random_state=42, n_jobs=-1,
        n_estimators=trial.suggest_int("n_estimators", 300, 999),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 4, 8),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 9),
    )
    return cv_rmse(model)


xgb_study = run_study("XGBoost", xgb_objective, n_trials=50)

### Does Optuna search better than the random search?

The best CV RMSE found so far, after every try. The dashed line is the result of the random search in notebook 02, the dotted line marks its number of tries.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, name in zip(axes, ["Random forest", "XGBoost"]):
    values = [t.value for t in studies[name].trials]
    best_so_far = np.minimum.accumulate(values)
    rs_score, rs_tries = random_search_02[name]

    ax.plot(range(1, len(values) + 1), values, "o", alpha=0.3, label="each try")
    ax.plot(range(1, len(values) + 1), best_so_far, label="Optuna: best so far")
    ax.axhline(rs_score, color="red", linestyle="--", label=f"random search (02): {rs_score}")
    ax.axvline(rs_tries, color="grey", linestyle=":", label=f"{rs_tries} tries")
    ax.set_ylim(best_so_far.min() - 3, rs_score + 15)
    ax.set_title(f"{name}: CV RMSE per try")
    ax.set_xlabel("try")
    ax.set_ylabel("CV RMSE")
    ax.legend()
plt.tight_layout()
plt.show()

pd.DataFrame({
    name: {
        "random search (02)": random_search_02[name][0],
        f"Optuna after {random_search_02[name][1]} tries": min(t.value for t in studies[name].trials[:random_search_02[name][1]]),
        f"Optuna after {len(studies[name].trials)} tries": studies[name].best_value,
    }
    for name in ["Random forest", "XGBoost"]
}).round(1)

**Finding:** _(filled in after the run)_

### CatBoost

In notebook 02, CatBoost ran with its default settings. Here three of its most important hyperparameters are tuned:
- `depth`: the depth of each tree (CatBoost builds symmetric trees, the same split on each level),
- `learning_rate`: how much of each new tree is added,
- `l2_leaf_reg`: a penalty on large leaf values, like `alpha` in Ridge.

`iterations=1000` stays fixed, as in notebook 02. Each try takes several seconds, so 25 tries.

In [ ]:
def cat_objective(trial):
    model = CatBoostRegressor(
        iterations=1000, random_seed=42, verbose=0,
        depth=trial.suggest_int("depth", 4, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 10, log=True),
    )
    return cv_rmse(model)


cat_study = run_study("CatBoost", cat_objective, n_trials=25)

### Voting and stacking with the tuned models

The same two ensembles as in notebook 02, built from the three models with Optuna's best settings.

In [ ]:
best_rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1, **rf_study.best_params)
best_xgb = XGBRegressor(random_state=42, n_jobs=-1, **xgb_study.best_params)
best_cat = CatBoostRegressor(iterations=1000, random_seed=42, verbose=0, **cat_study.best_params)
for model in (best_rf, best_xgb, best_cat):
    model.fit(X_train, y_train)

base_models = [("rf", best_rf), ("xgb", best_xgb), ("cat", best_cat)]
voting = VotingRegressor(base_models)
stacking = StackingRegressor(base_models, final_estimator=Ridge(), cv=kfold)

cv_voting = cv_rmse(voting)
cv_stacking = cv_rmse(stacking)
print(f"Voting   CV RMSE: {cv_voting:.1f}")
print(f"Stacking CV RMSE: {cv_stacking:.1f}")

# fit both on the whole training set for step 8
voting.fit(X_train, y_train)
stacking.fit(X_train, y_train)
print("stacking weights:", dict(zip(["rf", "xgb", "cat"], stacking.final_estimator_.coef_.round(2))))

### Model selection: notebook 02 vs. Optuna

In [ ]:
cv_04 = pd.Series({
    "Random forest": rf_study.best_value,
    "XGBoost": xgb_study.best_value,
    "CatBoost": cat_study.best_value,
    "Voting": cv_voting,
    "Stacking": cv_stacking,
})

cv_comparison = pd.DataFrame({"CV RMSE (02)": results_02["CV RMSE"], "CV RMSE (Optuna)": cv_04})
cv_comparison["change"] = cv_comparison["CV RMSE (Optuna)"] - cv_comparison["CV RMSE (02)"]
cv_comparison.round(1)

**Finding:** _(filled in after the run)_

## Step 8 – Test set

The test set is prepared with the same `prepare()` function and scored once, as in notebook 02.

In [ ]:
df_test_prep = prepare(df_test)
X_test = df_test_prep.drop(columns=["rentals", "datetime"])
y_test = df_test_prep["rentals"]

rows = []
for name, model in [("Random forest", best_rf), ("XGBoost", best_xgb), ("CatBoost", best_cat),
                    ("Voting", voting), ("Stacking", stacking)]:
    pred_test = model.predict(X_test)
    rows.append({
        "Model": name,
        "Train RMSE": root_mean_squared_error(y_train, model.predict(X_train)),
        "CV RMSE": cv_04[name],
        "Test RMSE": root_mean_squared_error(y_test, pred_test),
        "Test RMSE (02)": results_02.loc[name, "Test RMSE"],
        "Test MAE": mean_absolute_error(y_test, pred_test),
        "Test MAE (02)": results_02.loc[name, "Test MAE"],
        "Test R²": r2_score(y_test, pred_test),
    })

results_04 = pd.DataFrame(rows).set_index("Model")
results_04.round({"Train RMSE": 1, "CV RMSE": 1, "Test RMSE": 1, "Test RMSE (02)": 1,
                  "Test MAE": 1, "Test MAE (02)": 1, "Test R²": 3})

**Finding:** _(filled in after the run)_

## Conclusion: is Optuna worth it here?

_(filled in after the run)_